# LSTM CPD Replication

This notebook is the final presentation and reproducibility wrapper around
frozen implementation artifacts. It loads persisted outputs, imports existing
source modules, and intentionally keeps all core methodology outside notebook cells.

## Audience, Prerequisites, and Goal

This notebook is for readers who need a top-to-bottom view of the LSTM-CPD
replication pipeline after implementation artifacts have been frozen.

Prerequisites:
- the project root contains the frozen artifacts referenced below
- the `src/lstm_cpd` package is available from the project root
- this notebook is a wrapper only; no notebook-only core logic is allowed

Goal:
- surface the full pipeline in 12 ordered sections
- keep every result traceable to an artifact path or source module

## Outline

1. Implementation Contract
2. FTMO Data Contract
3. Canonical Daily-Close Layer
4. Base Features
5. CPD Outputs
6. Dataset Assembly
7. Model and Training Setup
8. Search Results
9. Selected Model
10. Causal Inference
11. Validation Evaluation
12. Reproducibility Manifest

In [1]:
from __future__ import annotations

import csv
import importlib
import json
import sys
from itertools import islice
from pathlib import Path


def _find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in (candidate, *candidate.parents):
        if (root / "src" / "lstm_cpd").exists() and (
            root / "spec_lstm_cpd_model_revised_sole_authority.md"
        ).exists():
            return root
    raise FileNotFoundError(
        "Unable to locate the project root from the notebook execution context."
    )


PROJECT_ROOT = _find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))


def _resolve_project_path(reference: str) -> Path:
    candidate = Path(reference)
    if candidate.is_absolute():
        return candidate
    return PROJECT_ROOT / candidate


def _preview_text(path: Path, *, max_lines: int = 8) -> str:
    with path.open("r", encoding="utf-8") as handle:
        lines = [line.rstrip("\n") for line in islice(handle, max_lines)]
    return "\n".join(lines)


def _preview_csv(path: Path, *, max_lines: int = 6) -> str:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return "".join(list(islice(handle, max_lines))).rstrip()


def _preview_json(path: Path) -> str:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(payload, dict):
        preview = {key: payload[key] for key in list(payload)[:5]}
    elif isinstance(payload, list):
        preview = payload[:2]
    else:
        preview = payload
    return json.dumps(preview, indent=2)


def preview_artifact(reference: str) -> str:
    path = _resolve_project_path(reference)
    suffix = path.suffix.lower()
    if suffix == ".json":
        return _preview_json(path)
    if suffix == ".csv":
        return _preview_csv(path)
    if suffix in {".md", ".txt", ".py"}:
        return _preview_text(path)
    size_bytes = path.stat().st_size
    return f"{path.name} ({size_bytes} bytes)"


def import_module_refs(module_refs: list[str]) -> list[tuple[str, str]]:
    resolved = []
    for module_name in module_refs:
        module = importlib.import_module(module_name)
        module_file = getattr(module, "__file__", None)
        module_path = "<built-in>" if module_file is None else str(Path(module_file).resolve())
        resolved.append((module_name, module_path))
    return resolved


def render_section(
    section_id: str,
    section_title: str,
    artifact_refs: list[str],
    module_refs: list[str],
) -> dict[str, object]:
    print(f"SECTION {section_id}: {section_title}")
    print(f"Project root: {PROJECT_ROOT}")
    print("")
    print("Artifacts")
    for artifact_ref in artifact_refs:
        print(f"- {artifact_ref}")
        print(preview_artifact(artifact_ref))
        print("")
    print("Modules")
    for module_name, module_path in import_module_refs(module_refs):
        print(f"- {module_name}: {module_path}")
    return {
        "section_id": section_id,
        "section_title": section_title,
        "artifact_refs": artifact_refs,
        "module_refs": module_refs,
    }

## 01. Implementation Contract

Freeze the authority documents that define the allowed methodology, execution-policy rules, and exclusions.

In [2]:
SECTION_ID = 'implementation_contract'
SECTION_TITLE = 'Implementation Contract'
ARTIFACT_REFS = ['spec_lstm_cpd_model_revised_sole_authority.md', 'lstm_cpd_implementation_plan.md', 'docs/contracts/invariant_ledger.md', 'docs/contracts/exclusions_ledger.md', 'docs/contracts/execution_policy_rules.md']
MODULE_REFS = []

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION implementation_contract: Implementation Contract
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- spec_lstm_cpd_model_revised_sole_authority.md
# Spec — LSTM CPD Model from *Slow Momentum with Fast Reversion*

## 0. Scope, provenance, and rules

This spec covers **only** the paper’s **LSTM + CPD** model: an online Gaussian-process changepoint detector whose outputs are appended to a Deep Momentum Network built around an LSTM, trained directly on a Sharpe-ratio objective.

This spec does **not** include:


- lstm_cpd_implementation_plan.md
# LSTM CPD Implementation Plan

## 1. Objective

Implement a spec-faithful Python replication of the paper’s **LSTM + CPD** model under `Dev/Research/Slow Momentum with fast mean reversion`, using only the project-defined FTMO sources, with the model logic fixed to: daily close-based arithmetic returns, 60-day EWM volatility, 5 normalized return features, 3 MACD 

{'section_id': 'implementation_contract',
 'section_title': 'Implementation Contract',
 'artifact_refs': ['spec_lstm_cpd_model_revised_sole_authority.md',
  'lstm_cpd_implementation_plan.md',
  'docs/contracts/invariant_ledger.md',
  'docs/contracts/exclusions_ledger.md',
  'docs/contracts/execution_policy_rules.md'],
 'module_refs': []}

## 02. FTMO Data Contract

Show the admissible FTMO universe, D-timeframe path resolution, and the schema and screening reports that bound the raw input layer.

In [3]:
SECTION_ID = 'ftmo_data_contract'
SECTION_TITLE = 'FTMO Data Contract'
ARTIFACT_REFS = ['artifacts/manifests/ftmo_asset_universe.json', 'artifacts/manifests/d_timeframe_path_manifest.json', 'docs/contracts/daily_close_schema_contract.md', 'artifacts/reports/schema_inspection_report.csv', 'artifacts/reports/asset_eligibility_report.csv', 'artifacts/reports/asset_exclusion_report.csv', 'artifacts/reports/minimum_history_screening_report.csv']
MODULE_REFS = ['lstm_cpd.ftmo_asset_universe', 'lstm_cpd.daily_close_contract', 'lstm_cpd.raw_history_screening']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION ftmo_data_contract: FTMO Data Contract
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/manifests/ftmo_asset_universe.json
[
  {
    "asset_id": "COCOA.c",
    "symbol": "COCOA.c",
    "category": "Agriculture"
  },
  {
    "asset_id": "COFFEE.c",
    "symbol": "COFFEE.c",
    "category": "Agriculture"
  }
]

- artifacts/manifests/d_timeframe_path_manifest.json
[
  {
    "asset_id": "COCOA.c",
    "symbol": "COCOA.c",
    "category": "Agriculture",
    "resolution_status": "RESOLVED",
    "resolution_failure_reason": null,
    "path_pattern": "category_symbol_d",
    "d_file_path": "data/FTMO Data/Agriculture/COCOA.c/D/COCOA.c_data.csv",
    "candidate_paths": [
      "data/FTMO Data/Agriculture/COCOA.c/D/COCOA.c_data.csv"
    ]
  },
  {
    "asset_id": "COFFEE.c",
    "symbol": "COFFEE.c",
    "category": "Agriculture",
    "resolution_status": "RESOLVED",
    "resolution_failure_reason

{'section_id': 'ftmo_data_contract',
 'section_title': 'FTMO Data Contract',
 'artifact_refs': ['artifacts/manifests/ftmo_asset_universe.json',
  'artifacts/manifests/d_timeframe_path_manifest.json',
  'docs/contracts/daily_close_schema_contract.md',
  'artifacts/reports/schema_inspection_report.csv',
  'artifacts/reports/asset_eligibility_report.csv',
  'artifacts/reports/asset_exclusion_report.csv',
  'artifacts/reports/minimum_history_screening_report.csv'],
 'module_refs': ['lstm_cpd.ftmo_asset_universe',
  'lstm_cpd.daily_close_contract',
  'lstm_cpd.raw_history_screening']}

## 03. Canonical Daily-Close Layer

Reference the frozen canonical close manifest that anchors all later feature and sequence work.

In [4]:
SECTION_ID = 'canonical_daily_close_layer'
SECTION_TITLE = 'Canonical Daily-Close Layer'
ARTIFACT_REFS = ['artifacts/manifests/canonical_daily_close_manifest.json']
MODULE_REFS = ['lstm_cpd.canonical_daily_close_store']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION canonical_daily_close_layer: Canonical Daily-Close Layer
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/manifests/canonical_daily_close_manifest.json
[
  {
    "asset_id": "COCOA.c",
    "symbol": "COCOA.c",
    "category": "Agriculture",
    "path_pattern": "category_symbol_d",
    "source_d_file_path": "data/FTMO Data/Agriculture/COCOA.c/D/COCOA.c_data.csv",
    "canonical_csv_path": "artifacts/canonical_daily_close/COCOA.c.csv",
    "row_count": 776,
    "first_timestamp": "2023-01-19T00:00:00",
    "last_timestamp": "2026-02-20T00:00:00",
    "file_hash": "sha256:fbd1504e552f4479d4ad97303d41bcb7758b6f17dd8e98ca44b70ac3a2655aef"
  },
  {
    "asset_id": "COFFEE.c",
    "symbol": "COFFEE.c",
    "category": "Agriculture",
    "path_pattern": "category_symbol_d",
    "source_d_file_path": "data/FTMO Data/Agriculture/COFFEE.c/D/COFFEE.c_data.csv",
    "canonical_csv_path": "artifacts/c

{'section_id': 'canonical_daily_close_layer',
 'section_title': 'Canonical Daily-Close Layer',
 'artifact_refs': ['artifacts/manifests/canonical_daily_close_manifest.json'],
 'module_refs': ['lstm_cpd.canonical_daily_close_store']}

## 04. Base Features

Summarize the deterministic return, volatility, normalized-return, MACD, and winsorization stack that produces the eight non-CPD features.

In [5]:
SECTION_ID = 'base_features'
SECTION_TITLE = 'Base Features'
ARTIFACT_REFS = ['artifacts/reports/feature_provenance_report.md']
MODULE_REFS = ['lstm_cpd.features.returns', 'lstm_cpd.features.volatility', 'lstm_cpd.features.normalized_returns', 'lstm_cpd.features.macd', 'lstm_cpd.features.winsorize']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION base_features: Base Features
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/reports/feature_provenance_report.md
# Feature Provenance Report

## Summary

- Generated asset count: 126
- Output artifact family: `artifacts/features/base/<asset_id>_base_features.csv`
- Output schema: `timestamp`, `asset_id`, five normalized returns, three MACD features


Modules


- lstm_cpd.features.returns: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/features/returns.py
- lstm_cpd.features.volatility: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/features/volatility.py
- lstm_cpd.features.normalized_returns: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/features/normalized_returns.py
- lstm_cpd.features.macd: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/features/macd.py
- lstm_cpd.features.winsorize: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/features/winsorize.py


{'section_id': 'base_features',
 'section_title': 'Base Features',
 'artifact_refs': ['artifacts/reports/feature_provenance_report.md'],
 'module_refs': ['lstm_cpd.features.returns',
  'lstm_cpd.features.volatility',
  'lstm_cpd.features.normalized_returns',
  'lstm_cpd.features.macd',
  'lstm_cpd.features.winsorize']}

## 05. CPD Outputs

Tie the GP-based CPD engine to its progress, telemetry, failure, fallback, and manifest artifacts.

In [6]:
SECTION_ID = 'cpd_outputs'
SECTION_TITLE = 'CPD Outputs'
ARTIFACT_REFS = ['docs/contracts/cpd_engine_contract.md', 'artifacts/reports/cpd_precompute_progress.csv', 'artifacts/reports/cpd_fit_telemetry.csv', 'artifacts/reports/cpd_failure_ledger.csv', 'artifacts/reports/cpd_fallback_ledger.csv', 'artifacts/manifests/cpd_feature_store_manifest.json']
MODULE_REFS = ['lstm_cpd.cpd.gp_kernels', 'lstm_cpd.cpd.fit_window', 'lstm_cpd.cpd.precompute', 'lstm_cpd.cpd.telemetry']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION cpd_outputs: CPD Outputs
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- docs/contracts/cpd_engine_contract.md
# CPD Engine Contract

This document is the binding output contract for `T-13`.

## Scope

- `T-13` provides the reusable single-window CPD fitting engine only.
- It does not materialize per-asset or per-LBW CPD CSVs. That remains `T-14`.

- artifacts/reports/cpd_precompute_progress.csv
lbw,asset_id,state,rows_written,last_timestamp,retry_count,fallback_count,started_at,finished_at,output_path,error_message
10,AAPL,completed,5330,2026-02-20T00:00:00,19,5,2026-04-23T08:08:46Z,2026-04-23T08:08:46Z,artifacts/features/cpd/lbw_10/AAPL_cpd.csv,
21,AAPL,completed,5330,2026-02-20T00:00:00,8,0,2026-04-23T22:38:09Z,2026-04-23T22:38:09Z,artifacts/features/cpd/lbw_21/AAPL_cpd.csv,
63,AAPL,completed,5330,2026-02-20T00:00:00,0,0,2026-04-24T20:00:16Z,2026-04-24T20:00:16Z,artifacts/features/cpd/lbw_63/A

[
  {
    "asset_id": "COCOA.c",
    "lbw": 10,
    "state": "present",
    "missing_reason": null,
    "cpd_csv_path": "artifacts/features/cpd/lbw_10/COCOA.c_cpd.csv",
    "row_count": 776,
    "canonical_row_count": 776,
    "first_timestamp": "2023-01-19T00:00:00",
    "last_timestamp": "2026-02-20T00:00:00",
    "output_row_count": 765,
    "retry_used_count": 1,
    "fallback_used_count": 0,
    "status_counts": {
      "success": 764,
      "retry_success": 1,
      "fallback_previous": 0,
      "baseline_failure": 0,
      "changepoint_failure": 0,
      "invalid_window": 11
    },
    "matches_canonical_timeline": true,
    "file_hash": "sha256:b116c2a14c587e8ad45f665fa93596ca05839cfd85166b1a2998082e5914b00c"
  },
  {
    "asset_id": "COCOA.c",
    "lbw": 21,
    "state": "present",
    "missing_reason": null,
    "cpd_csv_path": "artifacts/features/cpd/lbw_21/COCOA.c_cpd.csv",
    "row_count": 776,
    "canonical_row_count": 776,
    "first_timestamp": "2023-01-19T00:00:00",
 

- lstm_cpd.cpd.gp_kernels: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/cpd/gp_kernels.py
- lstm_cpd.cpd.fit_window: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/cpd/fit_window.py
- lstm_cpd.cpd.precompute: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/cpd/precompute.py
- lstm_cpd.cpd.telemetry: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/cpd/telemetry.py


{'section_id': 'cpd_outputs',
 'section_title': 'CPD Outputs',
 'artifact_refs': ['docs/contracts/cpd_engine_contract.md',
  'artifacts/reports/cpd_precompute_progress.csv',
  'artifacts/reports/cpd_fit_telemetry.csv',
  'artifacts/reports/cpd_failure_ledger.csv',
  'artifacts/reports/cpd_fallback_ledger.csv',
  'artifacts/manifests/cpd_feature_store_manifest.json'],
 'module_refs': ['lstm_cpd.cpd.gp_kernels',
  'lstm_cpd.cpd.fit_window',
  'lstm_cpd.cpd.precompute',
  'lstm_cpd.cpd.telemetry']}

## 06. Dataset Assembly

Reference the frozen dataset registry and the source modules that create joins, chronological splits, sequences, and array-ready registries.

In [7]:
SECTION_ID = 'dataset_assembly'
SECTION_TITLE = 'Dataset Assembly'
ARTIFACT_REFS = ['artifacts/manifests/dataset_registry.json']
MODULE_REFS = ['lstm_cpd.datasets.join_and_split', 'lstm_cpd.datasets.sequences', 'lstm_cpd.datasets.registry']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION dataset_assembly: Dataset Assembly
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/manifests/dataset_registry.json
[
  {
    "lbw": 10,
    "feature_columns": [
      "normalized_return_1",
      "normalized_return_21",
      "normalized_return_63",
      "normalized_return_126",
      "normalized_return_256",
      "macd_8_24",
      "macd_16_28",
      "macd_32_96",
      "nu",
      "gamma"
    ],
    "sequence_length": 63,
    "train_sequence_count": 4991,
    "val_sequence_count": 513,
    "train_input_shape": [
      4991,
      63,
      10
    ],
    "train_target_shape": [
      4991,
      63
    ],
    "val_input_shape": [
      513,
      63,
      10
    ],
    "val_target_shape": [
      513,
      63
    ],
    "artifacts": {
      "train_inputs_path": "artifacts/datasets/lbw_10/train_inputs.npy",
      "train_target_scale_path": "artifacts/datasets/lbw_10/train_target_sc

{'section_id': 'dataset_assembly',
 'section_title': 'Dataset Assembly',
 'artifact_refs': ['artifacts/manifests/dataset_registry.json'],
 'module_refs': ['lstm_cpd.datasets.join_and_split',
  'lstm_cpd.datasets.sequences',
  'lstm_cpd.datasets.registry']}

## 07. Model and Training Setup

Expose the shared LSTM runtime, Sharpe-loss contract, single-candidate runner, and the smoke-train fidelity artifacts.

In [8]:
SECTION_ID = 'model_training_setup'
SECTION_TITLE = 'Model and Training Setup'
ARTIFACT_REFS = ['docs/contracts/model_runtime_contract.md', 'artifacts/training/training_runner_contract.md', 'artifacts/training/smoke_run/smoke_config.json', 'artifacts/training/smoke_run/smoke_best_model.keras', 'artifacts/training/smoke_run/smoke_epoch_log.csv', 'artifacts/training/smoke_run/smoke_validation_history.csv', 'artifacts/reports/model_fidelity_report.md']
MODULE_REFS = ['lstm_cpd.model.network', 'lstm_cpd.training.losses', 'lstm_cpd.training.train_candidate']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION model_training_setup: Model and Training Setup
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- docs/contracts/model_runtime_contract.md
# Model Runtime Contract

## Purpose

This contract freezes the `T-19` model runtime used by every downstream candidate run.

## Runtime shape contract


- artifacts/training/training_runner_contract.md
# Training Runner Contract

## Purpose

This contract freezes the `T-20` single-candidate training runner.

## Entrypoint


- artifacts/training/smoke_run/smoke_config.json
{
  "candidate_id": "SMOKE",
  "candidate_index": 0,
  "dropout": 0.1,
  "hidden_size": 20,
  "minibatch_size": 64
}

- artifacts/training/smoke_run/smoke_best_model.keras
smoke_best_model.keras (37013 bytes)

- artifacts/training/smoke_run/smoke_epoch_log.csv
epoch_index,train_loss,val_loss,best_val_loss,mean_gradient_norm,improved
0,-0.1937866575183132,-0.32971063256263733,-0.3297106325626373

{'section_id': 'model_training_setup',
 'section_title': 'Model and Training Setup',
 'artifact_refs': ['docs/contracts/model_runtime_contract.md',
  'artifacts/training/training_runner_contract.md',
  'artifacts/training/smoke_run/smoke_config.json',
  'artifacts/training/smoke_run/smoke_best_model.keras',
  'artifacts/training/smoke_run/smoke_epoch_log.csv',
  'artifacts/training/smoke_run/smoke_validation_history.csv',
  'artifacts/reports/model_fidelity_report.md'],
 'module_refs': ['lstm_cpd.model.network',
  'lstm_cpd.training.losses',
  'lstm_cpd.training.train_candidate']}

## 08. Search Results

Display the immutable search schedule and the search summary report that collapses all candidate outcomes into a reviewable table.

In [9]:
SECTION_ID = 'search_results'
SECTION_TITLE = 'Search Results'
ARTIFACT_REFS = ['artifacts/training/search_schedule.json', 'artifacts/training/search_schedule.csv', 'artifacts/reports/search_summary_report.csv']
MODULE_REFS = ['lstm_cpd.training.search_schedule', 'lstm_cpd.training.search_runner']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION search_results: Search Results
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/training/search_schedule.json
[
  {
    "candidate_id": "C-001",
    "candidate_index": 0,
    "dropout": 0.3,
    "hidden_size": 5,
    "minibatch_size": 256,
    "learning_rate": 0.001,
    "max_grad_norm": 1.0,
    "lbw": 252
  },
  {
    "candidate_id": "C-002",
    "candidate_index": 1,
    "dropout": 0.5,
    "hidden_size": 40,
    "minibatch_size": 64,
    "learning_rate": 0.1,
    "max_grad_norm": 1.0,
    "lbw": 63
  }
]

- artifacts/training/search_schedule.csv
candidate_index,candidate_id,dropout,hidden_size,minibatch_size,learning_rate,max_grad_norm,lbw
0,C-001,0.3,5,256,0.001,1.0,252
1,C-002,0.5,40,64,0.1,1.0,63
2,C-003,0.2,80,256,0.01,100.0,21
3,C-004,0.1,160,128,0.01,100.0,63
4,C-005,0.3,5,64,0.0001,0.01,21

- artifacts/reports/search_summary_report.csv
candidate_index,candidate_id,status,selec

{'section_id': 'search_results',
 'section_title': 'Search Results',
 'artifact_refs': ['artifacts/training/search_schedule.json',
  'artifacts/training/search_schedule.csv',
  'artifacts/reports/search_summary_report.csv'],
 'module_refs': ['lstm_cpd.training.search_schedule',
  'lstm_cpd.training.search_runner']}

## 09. Selected Model

Show the winning-candidate metadata and the source helpers that resolve the selected checkpoint and candidate configuration.

In [10]:
SECTION_ID = 'selected_model'
SECTION_TITLE = 'Selected Model'
ARTIFACT_REFS = ['artifacts/training/best_candidate.json', 'artifacts/training/best_config.json']
MODULE_REFS = ['lstm_cpd.model_source', 'lstm_cpd.training.selection']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION selected_model: Selected Model
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/training/best_candidate.json
{
  "candidate_id": "C-031",
  "candidate_index": 30,
  "dropout": 0.5,
  "hidden_size": 20,
  "minibatch_size": 64
}

- artifacts/training/best_config.json
{
  "candidate_id": "C-031",
  "candidate_index": 30,
  "dropout": 0.5,
  "hidden_size": 20,
  "minibatch_size": 64
}

Modules
- lstm_cpd.model_source: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/model_source.py
- lstm_cpd.training.selection: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion/src/lstm_cpd/training/selection.py


{'section_id': 'selected_model',
 'section_title': 'Selected Model',
 'artifact_refs': ['artifacts/training/best_candidate.json',
  'artifacts/training/best_config.json'],
 'module_refs': ['lstm_cpd.model_source', 'lstm_cpd.training.selection']}

## 10. Causal Inference

Reference the latest causal sequence manifest and the next-day position outputs from the online inference path.

In [11]:
SECTION_ID = 'causal_inference'
SECTION_TITLE = 'Causal Inference'
ARTIFACT_REFS = ['artifacts/inference/latest_positions.csv', 'artifacts/inference/latest_sequence_manifest.csv']
MODULE_REFS = ['lstm_cpd.inference.online_inference']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION causal_inference: Causal Inference
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/inference/latest_positions.csv
asset_id,lbw,signal_timestamp,next_day_position,candidate_id,model_path
COCOA.c,63,2026-02-20T00:00:00,-0.011123972944915295,C-031,artifacts/training/candidates/candidate_030/best_model.keras
COFFEE.c,63,2026-02-20T00:00:00,0.069608069956302643,C-031,artifacts/training/candidates/candidate_030/best_model.keras
CORN.c,63,2026-02-23T00:00:00,-0.50008028745651245,C-031,artifacts/training/candidates/candidate_030/best_model.keras
SOYBEAN.c,63,2026-02-23T00:00:00,0.1439005434513092,C-031,artifacts/training/candidates/candidate_030/best_model.keras
WHEAT.c,63,2026-02-23T00:00:00,-0.058408800512552261,C-031,artifacts/training/candidates/candidate_030/best_model.keras

- artifacts/inference/latest_sequence_manifest.csv
asset_id,lbw,sequence_start_timestamp,sequence_end_timestamp,row

{'section_id': 'causal_inference',
 'section_title': 'Causal Inference',
 'artifact_refs': ['artifacts/inference/latest_positions.csv',
  'artifacts/inference/latest_sequence_manifest.csv'],
 'module_refs': ['lstm_cpd.inference.online_inference']}

## 11. Validation Evaluation

Show the raw and rescaled validation outputs and the report that explains the model-faithful FTMO-vs.-paper evaluation boundary.

In [12]:
SECTION_ID = 'validation_evaluation'
SECTION_TITLE = 'Validation Evaluation'
ARTIFACT_REFS = ['artifacts/evaluation/raw_validation_returns.csv', 'artifacts/evaluation/raw_validation_metrics.json', 'artifacts/evaluation/rescaled_validation_returns.csv', 'artifacts/evaluation/rescaled_validation_metrics.json', 'artifacts/evaluation/evaluation_report.md']
MODULE_REFS = ['lstm_cpd.evaluation.validation_evaluation']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION validation_evaluation: Validation Evaluation
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/evaluation/raw_validation_returns.csv
return_timestamp,asset_count,portfolio_return
2023-09-01T00:00:00,10,0.0023103549843654036
2023-09-04T00:00:00,11,-0.0010660859506143342
2023-09-05T00:00:00,12,-0.0015750591866587154
2023-09-06T00:00:00,12,-0.0003331551973436338
2023-09-07T00:00:00,13,-0.0015698917559348047

- artifacts/evaluation/raw_validation_metrics.json
{
  "annualized_return": 0.21438763520238976,
  "annualized_volatility": 0.21679880913575342,
  "annualized_downside_deviation": 0.15688449960167444,
  "sharpe_ratio": 0.9888782879252171,
  "sortino_ratio": 1.3665316570261195
}

- artifacts/evaluation/rescaled_validation_returns.csv
return_timestamp,asset_count,portfolio_return
2023-09-01T00:00:00,10,0.0015985016201717625
2023-09-04T00:00:00,11,-0.00073760964476523997
2023-09-05T00:00:00

{'section_id': 'validation_evaluation',
 'section_title': 'Validation Evaluation',
 'artifact_refs': ['artifacts/evaluation/raw_validation_returns.csv',
  'artifacts/evaluation/raw_validation_metrics.json',
  'artifacts/evaluation/rescaled_validation_returns.csv',
  'artifacts/evaluation/rescaled_validation_metrics.json',
  'artifacts/evaluation/evaluation_report.md'],
 'module_refs': ['lstm_cpd.evaluation.validation_evaluation']}

## 12. Reproducibility Manifest

Close the notebook with the manifest that hashes the selected model inputs and binds the frozen evaluation run back to the repo.

In [13]:
SECTION_ID = 'reproducibility_manifest'
SECTION_TITLE = 'Reproducibility Manifest'
ARTIFACT_REFS = ['artifacts/reproducibility/reproducibility_manifest.json']
MODULE_REFS = ['lstm_cpd.reproducibility.manifest']

render_section(SECTION_ID, SECTION_TITLE, ARTIFACT_REFS, MODULE_REFS)

SECTION reproducibility_manifest: Reproducibility Manifest
Project root: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mean reversion

Artifacts
- artifacts/reproducibility/reproducibility_manifest.json
{
  "run_id": "repro_c-031_lbw63",
  "run_type": "evaluation",
  "created_at_utc": "2026-04-28T11:16:12Z",
  "project_root": "Dev/Research/Slow Momentum with fast mean reversion",
  "authority_documents": {
    "spec": "spec_lstm_cpd_model_revised_sole_authority.md",
    "implementation_plan": "lstm_cpd_implementation_plan.md",
    "project_overview": "project_overview_slow_momentum_with_fast_reversion.md",
    "invariant_ledger": "docs/contracts/invariant_ledger.md",
    "exclusions_ledger": "docs/contracts/exclusions_ledger.md",
    "execution_policy_rules": "docs/contracts/execution_policy_rules.md"
  }
}

Modules
- lstm_cpd.reproducibility.manifest: /Users/bilalabdullajew/Arbeit/tradinglab/quantitive/Dev/Research/Slow Momentum with fast mea

{'section_id': 'reproducibility_manifest',
 'section_title': 'Reproducibility Manifest',
 'artifact_refs': ['artifacts/reproducibility/reproducibility_manifest.json'],
 'module_refs': ['lstm_cpd.reproducibility.manifest']}